In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0';

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


### Answers
1. We are trying to predict the sales price for each house in the dataset (i.e. the value of the SalePrice variable).

2. This is a supervised, offline, regression problem that uses data with mixed numeric and categorical features.

3. "Submissions are evaluated on Root-Mean-Squared-Error (RMSE) between the logarithm of the predicted value and the logarithm of the observed sales price," taken from the House Prices Kaggle Competition description. We're using log-RMSE because it ensures equal weight for expensive and cheap homes.




# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
# Libraries
import tensorflow as tf
import tensorflow_decision_forests as tfdf
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Comment this if the data visualisations doesn't work on your side
%matplotlib inline
print("TensorFlow v" + tf.__version__)
print("TensorFlow Decision Forests v" + tfdf.__version__)

print("\n")

In [ ]:
# Load Datasets
train_file_path = "../input/house-prices-advanced-regression-techniques/train.csv"
dataset_df = pd.read_csv(train_file_path)
print("Full train dataset shape is {}".format(dataset_df.shape))

print("\n")

# Inspecting the types of feature columns
#  Dropping the Id column only if it exists
if 'Id' in dataset_df.columns:
    dataset_df = dataset_df.drop('Id', axis=1)
dataset_df.info()

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
cat /kaggle/input/house-prices-advanced-regression-techniques/data_description.txt

In [ ]:
num_fts = dataset_df.select_dtypes(include=["int64", "float64"]).columns
print("int/float features: ", num_fts)

obj_fts = dataset_df.select_dtypes(include=["object"]).columns
print("catergorical features: ", obj_fts)

1. Features in the dataset come in all types:
    - categorical (e.g. MSSubClass, MSZoning, Street, Neighborhood, etc.)
    - ordinal (e.g. LotShape – going from "regular" to "irregular" – Landslope, OverallQual, etc.)
    - unbounded int/float (e.g. LotFrontage, LotArea, 1stFlrSF, 2ndFlrSF, etc.)
    - boolean/binary features (e.g. CentralAir, Street).

In [ ]:
null_count = dataset_df.isnull().sum()
null_count = null_count[null_count > 0]
print(f"Total columns with missing values: {len(null_count)}\n")
print(null_count.sort_values(ascending=False))

In [ ]:
dataset_df.head(3)

2. Features with missing values (where the total number of rows is 1460) are shown in the code block above. It is worth noting that the value `NaN`, which is inputted by design, is counted as a missing value. For example, houses with no basement or garage have values `NaN` despite not being truly missing.

In [ ]:
# Creating a new dataset with only numerical values
list(set(dataset_df.dtypes.tolist()))
df_num = dataset_df.select_dtypes(include = ['float64', 'int64'])
df_num.head()

In [ ]:
# Visualize the data using histograms
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

3. Using visual histograms, it was clear that many features have **outliers** because their ranges (i.e. y-axis values) are huge with no values in between. Features of this kind inlude `LotArea`, `BsmtFinSF1`, `BsmtFinSF2`, `BsmtHalfBath`, `KitchenAbvGr`, `WoodDeckSF`, `OpenPorchSF`, `Enclosed Porch`, `3SsnPorch`, `ScreenPorch`, `PoolArea`, and `MiscVal`.

4. Features that appear useful are the ones with
   - sufficient samples (i.e. few missing values),
   - informative distributions
   - relevance to target variable
   - low redundancy (i.e. is there another feature that provides the same insights?), and
   - low noise

Features of this kind include `LotFrontage`, `OverallQual`, `OverallCond`, `YearBuilt`, `BsmtFinSF1`, `GarageArea`.

Less useful or redundant features include `Id` (no predictive meaning), `Street`, `Alley`, and `MiscFeature` (sparse or binary with low variance). We also have features like `TotRmsAbvGrd` and `GrLivArea`, which are redundant.

In [ ]:
# Identify and characterize the target distribution.
dataset_df["SalePrice"].hist(bins=50)
plt.title("Distribution of SalePrice")
np.log1p(dataset_df["SalePrice"]).hist(bins=50)

5. The target variable `SalePrice` is a right-skewed continuous distribution, meaning most houses sell in mid-range prices with a long tail toward expensive homes.

In [ ]:
# Correlations between attributes
corr_matrix = dataset_df.corr(numeric_only=True)
corr_with_target = corr_matrix["SalePrice"].sort_values(ascending=False)
corr_with_target.head(10)

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel

# Divides target feature from the rest
assert "SalePrice" in dataset_df.columns
X, y = dataset_df.drop(columns=["SalePrice"]).copy(), dataset_df["SalePrice"].copy()
y_log = np.log1p(y)

In [ ]:
for c in ["FullBath","HalfBath","BsmtFullBath","BsmtHalfBath","TotalBsmtSF","1stFlrSF","2ndFlrSF"]:
    if c not in X: X[c] = 0
X[["FullBath","HalfBath","BsmtFullBath","BsmtHalfBath"]] = X[["FullBath","HalfBath","BsmtFullBath","BsmtHalfBath"]].fillna(0)
X["TotalBaths"]     = X["FullBath"] + 0.5*X["HalfBath"] + X["BsmtFullBath"] + 0.5*X["BsmtHalfBath"]
X["AgeSinceBuilt"]  = X.get("YrSold", np.nan) - X.get("YearBuilt", np.nan)
X["AgeSinceRemod"]  = X.get("YrSold", np.nan) - X.get("YearRemodAdd", np.nan)
X["TotalSF"]        = X["TotalBsmtSF"].fillna(0) + X["1stFlrSF"].fillna(0) + X["2ndFlrSF"].fillna(0)
X["IsRemodeled"]    = ((X.get("YearRemodAdd", np.nan) > X.get("YearBuilt", np.nan))).fillna(False).astype(int)

Summary of engineered features

- `TotalBaths`: weighted total number of bathrooms.
- `AgeSinceBuilt`: years since the house was built.
- `AgeSinceRemod`: years since last remodel.
- `TotalSF`: total floor area (basement + floors).
- `IsRemodeled`: 1 if the house was remodeled, else 0.

In [ ]:
# Groups of feature types
qual_order = ["Po","Fa","TA","Gd","Ex"]
ordinal_features = [c for c in ["ExterQual","KitchenQual","BsmtQual","HeatingQC"] if c in X.columns]
nominal_features = [c for c in ["Neighborhood","MSZoning","HouseStyle","SaleCondition","Exterior1st","Exterior2nd"] if c in X.columns]
num_selector = make_column_selector(dtype_include=np.number)
engineered_numeric = [c for c in ["TotalBaths","AgeSinceBuilt","AgeSinceRemod","TotalSF","IsRemodeled"] if c in X.columns]

# Pipelines
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                             ("scaler",  StandardScaler())])

ordinal_pipeline = ("drop" if not ordinal_features else
    Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
              ("ordenc",  OrdinalEncoder(categories=[qual_order]*len(ordinal_features),
                                         handle_unknown="use_encoded_value", unknown_value=-1))]))

categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                 ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# ColumnTransformer
all_numeric = list(set(num_selector(X)) | set(engineered_numeric))
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, all_numeric),
    ("ord", ordinal_pipeline, ordinal_features),
    ("cat", categorical_pipeline, nominal_features),
], remainder="drop")

# Full pipeline (clean → encode/scale → select)
preprocessing_pipeline = Pipeline([
    ("preprocess", preprocess),
    # This line performs feature selection based on how important each feature is to a Lasso regression model.
    ("feature_select", SelectFromModel(LassoCV(cv=5, n_alphas=50, random_state=42, max_iter=5000), threshold="median")),
])

# Fit/transform + glimpse
X_prep = preprocessing_pipeline.fit_transform(X, y)
print("Prepared shape:", X_prep.shape)

try:
    names = preprocessing_pipeline.named_steps["preprocess"].get_feature_names_out()
    mask  = preprocessing_pipeline.named_steps["feature_select"].get_support()
    if mask is not None and len(mask)==len(names): names = names[mask]
    print("Examples:", list(names)[:12])
except Exception:
    pass

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
# --- Imports ---
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# --- Define feature_names from the existing preprocessing_pipeline ---
try:
    names = preprocessing_pipeline.named_steps["preprocess"].get_feature_names_out()
    sel = preprocessing_pipeline.named_steps.get("feature_select", None)
    if sel is not None and hasattr(sel, "get_support"):
        mask = sel.get_support()
        if mask is not None and len(mask) == len(names):
            names = names[mask]
    feature_names = list(names)
except Exception:
    feature_names = None

# --- Models ---
models = {
    "LinearRegression (unregularized)": LinearRegression(),
    "Ridge (default alpha=1)": Ridge(random_state=42),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42)
}

# --- CV evaluation ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, mdl in models.items():
    rmse = np.sqrt(-cross_val_score(mdl, X_prep, y_log, scoring="neg_mean_squared_error", cv=cv, n_jobs=-1))
    results[name] = rmse
    print(f"{name}: RMSE(log) = {rmse.mean():.4f} ± {rmse.std():.4f}")

# --- Fit full-data models ---
fitted = {name: mdl.fit(X_prep, y_log) for name, mdl in models.items()}

# --- Top features (no helper funcs) ---
print("\n=== Top Features per Model ===")
for name, mdl in fitted.items():
    if hasattr(mdl, "coef_"):
        w = np.abs(np.ravel(mdl.coef_))
        idx = np.argsort(-w)[:10]
        print(f"\n{name}")
        for i in idx:
            label = feature_names[i] if feature_names and i < len(feature_names) else i
            print(f"{label}: {w[i]:.4f}")
    elif hasattr(mdl, "feature_importances_"):
        imp = mdl.feature_importances_
        idx = np.argsort(-imp)[:10]
        print(f"\n{name}")
        for i in idx:
            label = feature_names[i] if feature_names and i < len(feature_names) else i
            print(f"{label}: {imp[i]:.4f}")

# --- Hold-out split & pick best by CV mean ---
X_train, X_val, y_train, y_val = train_test_split(X_prep, y_log, test_size=0.2, random_state=42)
best_name = min(results, key=lambda k: results[k].mean())
best_model = fitted[best_name]

# --- Predictions & residuals ---
y_pred = best_model.predict(X_val)
residuals = y_val - y_pred


# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from scipy.stats import loguniform, uniform, randint
import numpy as np

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scorer = "neg_mean_squared_error"

# ---- ElasticNet (RandomizedSearchCV) ----
pipe_linear = Pipeline([("pre", preprocessing_pipeline),
                        ("model", ElasticNet(max_iter=10000, random_state=42))])

rs_linear = RandomizedSearchCV(
    pipe_linear,
    param_distributions={"model__alpha": loguniform(1e-4, 1e2),
                         "model__l1_ratio": uniform(0, 1)},
    n_iter=50, scoring=scorer, cv=cv, n_jobs=-1, random_state=42, verbose=0
)
rs_linear.fit(X, y_log)
best_linear = rs_linear.best_estimator_
best_linear_rmse = np.sqrt(-rs_linear.best_score_)

X_tr, X_va, y_tr, y_va = train_test_split(X, y_log, test_size=0.2, random_state=42)
best_linear.fit(X_tr, y_tr)
pred_va = best_linear.predict(X_va)
holdout_linear = np.sqrt(mean_squared_error(y_va, pred_va))

print("=== ElasticNet (RandomizedSearchCV) ===")
print("Best params:", rs_linear.best_params_)
print(f"CV RMSE (log): {best_linear_rmse:.4f} | Hold-out (log): {holdout_linear:.4f}")

# ---- Top ElasticNet features ----
names = best_linear.named_steps["pre"].named_steps["preprocess"].get_feature_names_out()
mask  = best_linear.named_steps["pre"].named_steps["feature_select"].get_support()
if mask is not None and len(mask)==len(names): names = names[mask]
coefs = np.abs(np.ravel(best_linear.named_steps["model"].coef_))
for i in np.argsort(-coefs)[:10]: print(f"{names[i]}: {coefs[i]:.4f}")

# ---- Decision Tree (RandomizedSearchCV, fast) ----
# Use lighter preprocessor (drop SelectFromModel for speed)
pre_tree = Pipeline([("preprocess", preprocess)])  # 'preprocess' from Step 4

pipe_tree = Pipeline([("pre", pre_tree),
                      ("model", DecisionTreeRegressor(random_state=42))])

rs_tree = RandomizedSearchCV(
    pipe_tree,
    param_distributions={
        "model__max_depth": randint(5, 31),
        "model__min_samples_split": randint(2, 21),
        "model__min_samples_leaf": randint(1, 11),
        "model__max_features": ["sqrt", "log2", 0.5],
        "model__ccp_alpha": uniform(0.0, 0.01),
    },
    n_iter=60, scoring=scorer, cv=KFold(n_splits=3, shuffle=True, random_state=42),
    n_jobs=-1, random_state=42, verbose=0
)
rs_tree.fit(X, y_log)
best_tree = rs_tree.best_estimator_
best_tree_rmse = np.sqrt(-rs_tree.best_score_)

best_tree.fit(X_tr, y_tr)
pred_va_tree = best_tree.predict(X_va)
holdout_tree = np.sqrt(mean_squared_error(y_va, pred_va_tree))

# --- define gs_tree for compatibility with existing print lines ---
gs_tree = rs_tree  # alias so gs_tree.best_params_ exists

print("\n=== Decision Tree (RandomizedSearchCV) ===")
print("Best params:", gs_tree.best_params_)
print(f"CV RMSE (log): {best_tree_rmse:.4f} | Hold-out (log): {holdout_tree:.4f}")

# ---- Top Tree features ----
names_tree = best_tree.named_steps["pre"].named_steps["preprocess"].get_feature_names_out()
imps = best_tree.named_steps["model"].feature_importances_
for i in np.argsort(-imps)[:10]: print(f"{names_tree[i]}: {imps[i]:.4f}")

In [ ]:
print("\n=== RESULTS SUMMARY ===")
print("ElasticNet →", rs_linear.best_params_,
      f"| CV: {best_linear_rmse:.4f} | Hold-out: {holdout_linear:.4f}")
print("Decision Tree →", gs_tree.best_params_,
      f"| CV: {best_tree_rmse:.4f} | Hold-out: {holdout_tree:.4f}")

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

### **Non-technical**
We built a pricing model that learns from dozens of property traits (overall quality, living area, garage capacity, and total square footage) to estimate sale prices consistently and fairly across cheap and expensive homes. Among the approaches we tried, the regularized linear model produced the most reliable accuracy, giving you a fast, defensible benchmark for listing, negotiating, and spotting under/over-priced homes.
### **Technical**
We implemented an end-to-end Scikit-Learn pipeline (imputation → engineered features: TotalBaths, TotalSF, AgeSinceBuilt, IsRemodeled → ordinal/one-hot encoding → scaling), tuned ElasticNet with RandomizedSearchCV and a Decision Tree with GridSearchCV, and evaluated with 5-fold CV using log-RMSE (matching Kaggle). ElasticNet outperformed the tuned tree on CV and a hold-out split; the most influential signals were OverallQual, GrLivArea, TotalSF, and garage capacity, with stable generalization.

# 9. Submit your predictions to the competition

In [ ]:
# --- Create correct test predictions & submission (log->price, predict on test) ---
import pandas as pd, numpy as np

# Load test
X_test_raw = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
ids = X_test_raw["Id"]

# Minimal feature engineering to match training (same as you did for X)
base = ["FullBath","HalfBath","BsmtFullBath","BsmtHalfBath","TotalBsmtSF","1stFlrSF","2ndFlrSF"]
for c in base:
    if c not in X_test_raw: X_test_raw[c] = 0
X_test_raw[base[:4]] = X_test_raw[base[:4]].fillna(0)
X_test_raw["TotalBaths"]    = X_test_raw["FullBath"] + 0.5*X_test_raw["HalfBath"] + X_test_raw["BsmtFullBath"] + 0.5*X_test_raw["BsmtHalfBath"]
X_test_raw["AgeSinceBuilt"] = X_test_raw.get("YrSold", np.nan) - X_test_raw.get("YearBuilt", np.nan)
X_test_raw["AgeSinceRemod"] = X_test_raw.get("YrSold", np.nan) - X_test_raw.get("YearRemodAdd", np.nan)
X_test_raw["TotalSF"]       = X_test_raw["TotalBsmtSF"].fillna(0) + X_test_raw["1stFlrSF"].fillna(0) + X_test_raw["2ndFlrSF"].fillna(0)
X_test_raw["IsRemodeled"]   = ((X_test_raw.get("YearRemodAdd", np.nan) > X_test_raw.get("YearBuilt", np.nan))).fillna(False).astype(int)

# Refit chosen model on all training data (model outputs log-price)
final_model = best_linear if holdout_linear <= holdout_tree else best_tree
final_model.fit(X, y_log)

# Predict on test (log space) -> clip -> expm1 to recover SalePrice
y_test_log = final_model.predict(X_test_raw)
test_predictions = np.expm1(np.clip(y_test_log, 1.0, None))

# Save submission
pd.DataFrame({"Id": ids, "SalePrice": test_predictions}).to_csv("submission.csv", index=False)

In [ ]:
test_predictions

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)

In [ ]:
!ls -lh submission.csv
!head -n 5 submission.csv

In [ ]:
%%bash

cat submission.csv | head -n 5